In [0]:
--crosswalk 
DESCRIBE TABLE pondolens.dev_silver.shared_region_crosswalk;

SELECT *
FROM pondolens.dev_silver.shared_region_crosswalk
ORDER BY source_region;

SELECT
  source_region,
  region_code,
  COUNT(*) AS row_count
FROM pondolens.dev_silver.shared_region_crosswalk
GROUP BY source_region, region_code
ORDER BY source_region, region_code;


CREATE OR REPLACE TABLE pondolens.dev_silver.shared_region_crosswalk AS
SELECT *
FROM VALUES
  ('BARMM', 'BARMM', 'BARMM'),
  ('Cordillera Administrative Region', 'CAR', 'Cordillera Administrative Region'),
  ('MIMAROPA Region', 'IVB', 'MIMAROPA'),
  ('NCR - National Capital Region', 'NCR', 'National Capital Region'),
  ('NCR National Capital Region', 'NCR', 'National Capital Region'),
  ('Region I - Ilocos Region', 'I', 'Ilocos Region'),
  ('Region I Ilocos Region', 'I', 'Ilocos Region'),
  ('Region II - Cagayan Valley', 'II', 'Cagayan Valley'),
  ('Region II Cagayan Valley', 'II', 'Cagayan Valley'),
  ('Region III - Central Luzon', 'III', 'Central Luzon'),
  ('Region III Central Luzon', 'III', 'Central Luzon'),
  ('Region IV A CALABARZON', 'IVA', 'CALABARZON'),
  ('Region IV-A CALABARZON', 'IVA', 'CALABARZON'),
  ('Region V - Bicol Region', 'V', 'Bicol Region'),
  ('Region V Bicol Region', 'V', 'Bicol Region'),
  ('Region VI - Western Visayas', 'VI', 'Western Visayas'),
  ('Region VI Western Visayas', 'VI', 'Western Visayas'),
  ('Region VII - Central Visayas', 'VII', 'Central Visayas'),
  ('Region VII Central Visayas', 'VII', 'Central Visayas'),
  ('Region VIII - Eastern Visayas', 'VIII', 'Eastern Visayas'),
  ('Region VIII Eastern Visayas', 'VIII', 'Eastern Visayas'),
  ('Region IX - Zamboanga Peninsula', 'IX', 'Zamboanga Peninsula'),
  ('Region IX Zamboanga Peninsula', 'IX', 'Zamboanga Peninsula'),
  ('Region X - Northern Mindanao', 'X', 'Northern Mindanao'),
  ('Region X Northern Mindanao', 'X', 'Northern Mindanao'),
  ('Region XI - Davao Region', 'XI', 'Davao Region'),
  ('Region XI Davao Region', 'XI', 'Davao Region'),
  ('Region XII - SOCCSKSARGEN', 'XII', 'SOCCSKSARGEN'),
  ('Region XII SOCCSKSARGEN', 'XII', 'SOCCSKSARGEN'),
  ('Region XIII - Caraga', 'XIII', 'Caraga'),
  ('Region XIII Caraga', 'XIII', 'Caraga')
AS crosswalk(source_region_name, region_code, region_name);


SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT source_region_name) AS source_names,
  COUNT(DISTINCT region_code) AS region_codes,
  COUNT(DISTINCT region_name) AS standard_region_names
FROM pondolens.dev_silver.shared_region_crosswalk;


SELECT *
FROM pondolens.dev_silver.shared_region_crosswalk
ORDER BY region_code, source_region_name;


DESCRIBE TABLE pondolens.dev_bronze.hospital;


SELECT
  h.region_name AS source_region_name,
  x.region_code,
  x.region_name AS standard_region_name,
  COUNT(*) AS row_count,
  SUM(CASE WHEN TRY_CAST(TRIM(h.year) AS INT) IS NULL THEN 1 ELSE 0 END)
    AS invalid_year_rows,
  SUM(CASE WHEN TRY_CAST(TRIM(h.regulated_hospital_count) AS INT) IS NULL THEN 1 ELSE 0 END)
    AS invalid_count_rows
FROM pondolens.dev_bronze.hospital h
LEFT JOIN pondolens.dev_silver.shared_region_crosswalk x
  ON TRIM(h.region_name) = x.source_region_name
GROUP BY
  h.region_name,
  x.region_code,
  x.region_name
ORDER BY
  source_region_name;

SELECT
  year,
  COUNT(*) AS row_count,
  COUNT(DISTINCT region_name) AS region_count,
  COLLECT_SET(hospital_type) AS hospital_types,
  COLLECT_SET(unit) AS units
FROM pondolens.dev_bronze.hospital
GROUP BY year
ORDER BY year;


SELECT
  year,
  region_name,
  hospital_type,
  COUNT(*) AS duplicate_count
FROM pondolens.dev_bronze.hospital
GROUP BY year, region_name, hospital_type
HAVING COUNT(*) > 1
ORDER BY year, region_name, hospital_type;

SELECT
  source_region_name,
  COUNT(*) AS row_count,
  COUNT(DISTINCT region_code) AS region_code_count,
  COUNT(DISTINCT region_name) AS region_name_count
FROM pondolens.dev_silver.shared_region_crosswalk
GROUP BY source_region_name
HAVING COUNT(*) <> 1
    OR COUNT(DISTINCT region_code) <> 1
    OR COUNT(DISTINCT region_name) <> 1;



------------------------------------------------------------------
--Query for Hospital

CREATE OR REPLACE TABLE pondolens.dev_silver.hospital AS
WITH cleaned AS (
  SELECT
    TRY_CAST(TRIM(h.year) AS INT) AS year,
    x.region_code,
    x.region_name,
    TRIM(h.hospital_type) AS hospital_type,
    TRY_CAST(TRIM(h.regulated_hospital_count) AS INT)
      AS regulated_hospital_count,
    LOWER(TRIM(h.unit)) AS unit,
    TRIM(h.source_agency) AS source_agency,
    TRIM(h.source_publication) AS source_publication,
    TRIM(h.source_table) AS source_table,
    TRIM(h.source_page) AS source_page,
    TRIM(h.source_url) AS source_url,
    TRIM(h.coverage_note) AS coverage_note
  FROM pondolens.dev_bronze.hospital h
  INNER JOIN pondolens.dev_silver.shared_region_crosswalk x
    ON TRIM(h.region_name) = x.source_region_name
  WHERE TRY_CAST(TRIM(h.year) AS INT) IS NOT NULL
    AND TRY_CAST(TRIM(h.regulated_hospital_count) AS INT) IS NOT NULL
)
SELECT
  CAST(
    7000000 + ROW_NUMBER() OVER (
      ORDER BY year, region_code, hospital_type
    ) AS INT
  ) AS hospital_id,
  year,
  region_code,
  region_name,
  hospital_type,
  regulated_hospital_count,
  unit,
  source_agency,
  source_publication,
  source_table,
  source_page,
  source_url,
  coverage_note,
  CURRENT_TIMESTAMP() AS processed_at
FROM cleaned;



SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT hospital_id) AS unique_ids,
  COUNT(DISTINCT year) AS year_count,
  COUNT(DISTINCT region_code) AS region_count,
  MIN(year) AS first_year,
  MAX(year) AS last_year,
  SUM(CASE WHEN hospital_id IS NULL THEN 1 ELSE 0 END) AS null_ids,
  SUM(CASE WHEN region_code IS NULL THEN 1 ELSE 0 END) AS null_region_codes,
  SUM(CASE WHEN regulated_hospital_count IS NULL THEN 1 ELSE 0 END)
    AS null_hospital_counts
FROM pondolens.dev_silver.hospital;


SELECT
  year,
  region_code,
  hospital_type,
  COUNT(*) AS row_count
FROM pondolens.dev_silver.hospital
GROUP BY year, region_code, hospital_type
HAVING COUNT(*) > 1;


WITH bronze AS (
  SELECT
    TRY_CAST(TRIM(year) AS INT) AS year,
    COUNT(*) AS bronze_rows,
    SUM(TRY_CAST(TRIM(regulated_hospital_count) AS BIGINT))
      AS bronze_hospital_count
  FROM pondolens.dev_bronze.hospital
  GROUP BY TRY_CAST(TRIM(year) AS INT)
),
silver AS (
  SELECT
    year,
    COUNT(*) AS silver_rows,
    SUM(regulated_hospital_count) AS silver_hospital_count
  FROM pondolens.dev_silver.hospital
  GROUP BY year
)
SELECT
  COALESCE(b.year, s.year) AS year,
  b.bronze_rows,
  s.silver_rows,
  s.silver_rows - b.bronze_rows AS count_difference,
  b.bronze_hospital_count,
  s.silver_hospital_count,
  s.silver_hospital_count - b.bronze_hospital_count
    AS hospital_count_difference
FROM bronze b
FULL OUTER JOIN silver s
  ON b.year = s.year
ORDER BY year;


